# CA binding

In [124]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import confusion_matrix, matthews_corrcoef

# Load CSVs
df_user = pd.read_csv('CA_metal_per_protein_predictions.csv')
df_lms = pd.read_csv('CA_Test.bind.csv', skiprows=1)

# First, compute threshold A: min prob where pred==1 across all proteins
all_pos_probs = []
for _, row in df_user.iterrows():
    user_probs = [float(p) for p in row['prob'].split(',')]
    user_preds = [int(p) for p in row['pred'].split(',')]
    pos_probs = [user_probs[i] for i in range(len(user_preds)) if user_preds[i] == 1]
    all_pos_probs.extend(pos_probs)

A = min(all_pos_probs) if all_pos_probs else 0.5  # fallback if no positives
A= 0.34
print(f"Computed threshold A: {A}")

# Now, accumulate global y_true and y_pred_new
global_y_true = []
global_y_pred_new = []

for _, row in df_user.iterrows():
    pid = row['ID']
    seq_len = len(row['sequence'])
    
    # Parse user's data
    tp_list = ast.literal_eval(row['TP'])
    fp_list = ast.literal_eval(row['FP'])
    tn_list = ast.literal_eval(row['TN'])
    fn_list = ast.literal_eval(row['FN'])
    user_probs = [float(p) for p in row['prob'].split(',')]
    
    # Reconstruct y_true
    y_true_local = np.zeros(seq_len)
    y_true_local[tp_list + fn_list] = 1
    global_y_true.extend(y_true_local)
    
    # Get LMetalSite probs for Ca2+
    lms_row = df_lms[df_lms['ID'] == pid]
    if lms_row.empty:
        print(f"Warning: No LMetalSite data for {pid}, skipping")
        continue
    lms_probs_str = lms_row['Ca2+ binding prob'].values[0]
    lms_probs = [float(p.strip()) for p in lms_probs_str.split(',')]
    
    # Positive positions (TP + FP)
    pos_idx = tp_list + fp_list
    
    # New predictions: start with 0 everywhere
    y_pred_new_local = np.zeros(seq_len)
    
    # Only update positives
    for i in pos_idx:
        if i < len(lms_probs):  # safety
            avg_prob = (user_probs[i]*0.3 + lms_probs[i]*0.7)
            # avg_prob = lms_probs[i]
            if avg_prob >= A:
                y_pred_new_local[i] = 1
    
    global_y_pred_new.extend(y_pred_new_local)

# Convert to arrays
global_y_true = np.array(global_y_true)
global_y_pred_new = np.array(global_y_pred_new)

# Compute global confusion matrix
if len(global_y_true) > 0:
    TN, FP, FN, TP = confusion_matrix(global_y_true, global_y_pred_new).ravel()
    Sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    Spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    Acc = (TP + TN) / (TP + FP + TN + FN)
    Precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    F1 = 2 * Precision * Sens / (Precision + Sens) if (Precision + Sens) > 0 else 0.0
    MCC = matthews_corrcoef(global_y_true, global_y_pred_new)
    
    print(f"New TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
    print(f"Sensitivity: {Sens:.4f}, Specificity: {Spec:.4f}, Accuracy: {Acc:.4f}")
    print(f"Precision: {Precision:.4f}, F1: {F1:.4f}, MCC: {MCC:.4f}")
else:
    print("No data processed")

Computed threshold A: 0.34
New TP: 391, FP: 122, TN: 65698, FN: 643
Sensitivity: 0.3781, Specificity: 0.9981, Accuracy: 0.9886
Precision: 0.7622, F1: 0.5055, MCC: 0.5321


In [40]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import confusion_matrix, matthews_corrcoef

# ────────────────────────────────────────────────────────────────
# Settings — same as before
# ────────────────────────────────────────────────────────────────
USER_WEIGHT = 0.1
LMS_WEIGHT  = 0.9
# A = 0.285
# A = 0.34
A = 0.045
df_user = pd.read_csv('CA_metal_per_protein_predictions.csv')
df_lms  = pd.read_csv('CA_Test.bind.csv', skiprows=1)

print(f"Real-world inference mode: filter ALL candidate positives")
print(f"Weights → User: {USER_WEIGHT:.2f} | LMetalSite: {LMS_WEIGHT:.2f}")
print(f"Threshold A = {A}")

global_y_true = []          # only used if labels exist (for evaluation)
global_y_pred_new = []

for _, row in df_user.iterrows():
    pid = row['ID']
    sequence = row['sequence']
    seq_len = len(sequence)

    # Parse your model's predictions
    user_probs = [float(p) for p in row['prob'].split(',')]
    user_preds = [int(p) for p in row['pred'].split(',')]

    # If labels exist (for evaluation only)
    has_labels = 'TP' in row and 'FP' in row
    if has_labels:
        tp_list = ast.literal_eval(row['TP'])
        fn_list = ast.literal_eval(row['FN'])
        y_true_local = np.zeros(seq_len)
        y_true_local[tp_list + fn_list] = 1
        global_y_true.extend(y_true_local)
    else:
        y_true_local = None  # no labels → real prediction mode

    # Get LMetalSite probs
    lms_probs = None
    lms_row = df_lms[df_lms['ID'] == pid]
    if not lms_row.empty:
        lms_probs_str = lms_row['Ca2+ binding prob'].values[0]
        lms_probs = [float(p.strip()) for p in lms_probs_str.split(',')]

    # ── Real inference: filter ALL candidate positions ──
    y_pred_new_local = np.zeros(seq_len)

    for i in range(seq_len):
        if user_preds[i] == 0:
            continue  # not a candidate → skip

        # This is a candidate (your model predicted positive)
        avg_prob = user_probs[i]  # default: at least your prob
        if lms_probs is not None and i < len(lms_probs):
            avg_prob = USER_WEIGHT * user_probs[i] + LMS_WEIGHT * lms_probs[i]

        if avg_prob >= A:
            y_pred_new_local[i] = 1

    global_y_pred_new.extend(y_pred_new_local)

# ────────────────────────────────────────────────────────────────
# Evaluation (only if labels were present)
# ────────────────────────────────────────────────────────────────
global_y_true = np.array(global_y_true) if global_y_true else None
global_y_pred_new = np.array(global_y_pred_new)

if global_y_true is not None and len(global_y_true) > 0:
    TN, FP, FN, TP = confusion_matrix(global_y_true, global_y_pred_new).ravel()

    Sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    Spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    Acc  = (TP + TN) / len(global_y_true)
    Prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    F1   = 2 * Prec * Sens / (Prec + Sens) if (Prec + Sens) > 0 else 0.0
    MCC  = matthews_corrcoef(global_y_true, global_y_pred_new)

    print("\nEvaluation on labeled test set (simulating real inference):")
    print(f"  TP: {TP:6d}")
    print(f"  FP: {FP:6d}")
    print(f"  TN: {TN:6d}")
    print(f"  FN: {FN:6d}")
    print(f"Sensitivity: {Sens:.4f}")
    print(f"Specificity: {Spec:.4f}")
    print(f"Accuracy:    {Acc:.4f}")
    print(f"Precision:   {Prec:.4f}")
    print(f"F1-score:    {F1:.4f}")
    print(f"MCC:         {MCC:.4f}")
else:
    print("\nReal prediction mode: no labels available → only final predictions generated.")

Real-world inference mode: filter ALL candidate positives
Weights → User: 0.10 | LMetalSite: 0.90
Threshold A = 0.045

Evaluation on labeled test set (simulating real inference):
  TP:    566
  FP:   2770
  TN:  63050
  FN:    468
Sensitivity: 0.5474
Specificity: 0.9579
Accuracy:    0.9516
Precision:   0.1697
F1-score:    0.2590
MCC:         0.2864


# MG Binding

In [24]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import confusion_matrix, matthews_corrcoef

# Load CSVs
df_user = pd.read_csv('MG_metal_per_protein_predictions.csv')
df_lms = pd.read_csv('MG_Test.bind.csv', skiprows=1)

# First, compute threshold A: min prob where pred==1 across all proteins
all_pos_probs = []
for _, row in df_user.iterrows():
    user_probs = [float(p) for p in row['prob'].split(',')]
    user_preds = [int(p) for p in row['pred'].split(',')]
    pos_probs = [user_probs[i] for i in range(len(user_preds)) if user_preds[i] == 1]
    all_pos_probs.extend(pos_probs)

A = min(all_pos_probs) if all_pos_probs else 0.5  # fallback if no positives
A = 0.006348
print(f"Computed threshold A: {A}")

# Now, accumulate global y_true and y_pred_new
global_y_true = []
global_y_pred_new = []

for _, row in df_user.iterrows():
    pid = row['ID']
    seq_len = len(row['sequence'])
    
    # Parse user's data
    tp_list = ast.literal_eval(row['TP'])
    fp_list = ast.literal_eval(row['FP'])
    tn_list = ast.literal_eval(row['TN'])
    fn_list = ast.literal_eval(row['FN'])
    user_probs = [float(p) for p in row['prob'].split(',')]
    
    # Reconstruct y_true
    y_true_local = np.zeros(seq_len)
    y_true_local[tp_list + fn_list] = 1
    global_y_true.extend(y_true_local)
    
    # Get LMetalSite probs for Ca2+
    lms_row = df_lms[df_lms['ID'] == pid]
    if lms_row.empty:
        print(f"Warning: No LMetalSite data for {pid}, skipping")
        continue
    lms_probs_str = lms_row['Mg2+ binding prob'].values[0]
    lms_probs = [float(p.strip()) for p in lms_probs_str.split(',')]
    
    # Positive positions (TP + FP)
    pos_idx = tp_list + fp_list
    
    # New predictions: start with 0 everywhere
    y_pred_new_local = np.zeros(seq_len)
    
    # Only update positives
    for i in pos_idx:
        if i < len(lms_probs):  # safety
            avg_prob = (user_probs[i]*0.1 + lms_probs[i]*0.9)
            # avg_prob = lms_probs[i]
            if avg_prob >= A:
                y_pred_new_local[i] = 1
    
    global_y_pred_new.extend(y_pred_new_local)

# Convert to arrays
global_y_true = np.array(global_y_true)
global_y_pred_new = np.array(global_y_pred_new)

# Compute global confusion matrix
if len(global_y_true) > 0:
    TN, FP, FN, TP = confusion_matrix(global_y_true, global_y_pred_new).ravel()
    Sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    Spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    Acc = (TP + TN) / (TP + FP + TN + FN)
    Precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    F1 = 2 * Precision * Sens / (Precision + Sens) if (Precision + Sens) > 0 else 0.0
    MCC = matthews_corrcoef(global_y_true, global_y_pred_new)
    
    print(f"New TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
    print(f"Sensitivity: {Sens:.4f}, Specificity: {Spec:.4f}, Accuracy: {Acc:.4f}")
    print(f"Precision: {Precision:.4f}, F1: {F1:.4f}, MCC: {MCC:.4f}")
else:
    print("No data processed")

Computed threshold A: 0.008569
New TP: 574, FP: 10012, TN: 77901, FN: 319
Sensitivity: 0.6428, Specificity: 0.8861, Accuracy: 0.8837
Precision: 0.0542, F1: 0.1000, MCC: 0.1629


In [34]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import confusion_matrix, matthews_corrcoef

# ────────────────────────────────────────────────────────────────
# Settings — same as before
# ────────────────────────────────────────────────────────────────
USER_WEIGHT = 0.10
LMS_WEIGHT  = 0.90
# A = 0.5
A = 0.01
df_user = pd.read_csv('MG_metal_per_protein_predictions.csv')
df_lms  = pd.read_csv('MG_Test.bind.csv', skiprows=1)

print(f"Real-world inference mode: filter ALL candidate positives")
print(f"Weights → User: {USER_WEIGHT:.2f} | LMetalSite: {LMS_WEIGHT:.2f}")
print(f"Threshold A = {A}")

global_y_true = []          # only used if labels exist (for evaluation)
global_y_pred_new = []

for _, row in df_user.iterrows():
    pid = row['ID']
    sequence = row['sequence']
    seq_len = len(sequence)

    # Parse your model's predictions
    user_probs = [float(p) for p in row['prob'].split(',')]
    user_preds = [int(p) for p in row['pred'].split(',')]

    # If labels exist (for evaluation only)
    has_labels = 'TP' in row and 'FP' in row
    if has_labels:
        tp_list = ast.literal_eval(row['TP'])
        fn_list = ast.literal_eval(row['FN'])
        y_true_local = np.zeros(seq_len)
        y_true_local[tp_list + fn_list] = 1
        global_y_true.extend(y_true_local)
    else:
        y_true_local = None  # no labels → real prediction mode

    # Get LMetalSite probs
    lms_probs = None
    lms_row = df_lms[df_lms['ID'] == pid]
    if not lms_row.empty:
        lms_probs_str = lms_row['Mg2+ binding prob'].values[0]
        lms_probs = [float(p.strip()) for p in lms_probs_str.split(',')]

    # ── Real inference: filter ALL candidate positions ──
    y_pred_new_local = np.zeros(seq_len)

    for i in range(seq_len):
        if user_preds[i] == 0:
            continue  # not a candidate → skip

        # This is a candidate (your model predicted positive)
        avg_prob = user_probs[i]  # default: at least your prob
        if lms_probs is not None and i < len(lms_probs):
            avg_prob = USER_WEIGHT * user_probs[i] + LMS_WEIGHT * lms_probs[i]

        if avg_prob >= A:
            y_pred_new_local[i] = 1

    global_y_pred_new.extend(y_pred_new_local)

# ────────────────────────────────────────────────────────────────
# Evaluation (only if labels were present)
# ────────────────────────────────────────────────────────────────
global_y_true = np.array(global_y_true) if global_y_true else None
global_y_pred_new = np.array(global_y_pred_new)

if global_y_true is not None and len(global_y_true) > 0:
    TN, FP, FN, TP = confusion_matrix(global_y_true, global_y_pred_new).ravel()

    Sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    Spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    Acc  = (TP + TN) / len(global_y_true)
    Prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    F1   = 2 * Prec * Sens / (Prec + Sens) if (Prec + Sens) > 0 else 0.0
    MCC  = matthews_corrcoef(global_y_true, global_y_pred_new)

    print("\nEvaluation on labeled test set (simulating real inference):")
    print(f"  TP: {TP:6d}")
    print(f"  FP: {FP:6d}")
    print(f"  TN: {TN:6d}")
    print(f"  FN: {FN:6d}")
    print(f"Sensitivity: {Sens:.4f}")
    print(f"Specificity: {Spec:.4f}")
    print(f"Accuracy:    {Acc:.4f}")
    print(f"Precision:   {Prec:.4f}")
    print(f"F1-score:    {F1:.4f}")
    print(f"MCC:         {MCC:.4f}")
else:
    print("\nReal prediction mode: no labels available → only final predictions generated.")

Real-world inference mode: filter ALL candidate positives
Weights → User: 0.10 | LMetalSite: 0.90
Threshold A = 0.01

Evaluation on labeled test set (simulating real inference):
  TP:    419
  FP:   3293
  TN:  84620
  FN:    474
Sensitivity: 0.4692
Specificity: 0.9625
Accuracy:    0.9576
Precision:   0.1129
F1-score:    0.1820
MCC:         0.2152


# MN Binding

In [16]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import confusion_matrix, matthews_corrcoef

# ────────────────────────────────────────────────────────────────
# Settings — same as before
# ────────────────────────────────────────────────────────────────
USER_WEIGHT = 0.1
LMS_WEIGHT  = 0.9
# A = 0.24803199999999997
A = 0.026064
# A = 0.47 # for Mn ion
df_user = pd.read_csv('MN_metal_per_protein_predictions.csv')
df_lms  = pd.read_csv('MN_Test.bind.csv', skiprows=1)

print(f"Real-world inference mode: filter ALL candidate positives")
print(f"Weights → User: {USER_WEIGHT:.2f} | LMetalSite: {LMS_WEIGHT:.2f}")
print(f"Threshold A = {A}")

global_y_true = []          # only used if labels exist (for evaluation)
global_y_pred_new = []

for _, row in df_user.iterrows():
    pid = row['ID']
    sequence = row['sequence']
    seq_len = len(sequence)

    # Parse your model's predictions
    user_probs = [float(p) for p in row['prob'].split(',')]
    user_preds = [int(p) for p in row['pred'].split(',')]

    # If labels exist (for evaluation only)
    has_labels = 'TP' in row and 'FP' in row
    if has_labels:
        tp_list = ast.literal_eval(row['TP'])
        fn_list = ast.literal_eval(row['FN'])
        y_true_local = np.zeros(seq_len)
        y_true_local[tp_list + fn_list] = 1
        global_y_true.extend(y_true_local)
    else:
        y_true_local = None  # no labels → real prediction mode

    # Get LMetalSite probs
    lms_probs = None
    lms_row = df_lms[df_lms['ID'] == pid]
    if not lms_row.empty:
        lms_probs_str = lms_row['Mn2+ binding prob'].values[0]
        lms_probs = [float(p.strip()) for p in lms_probs_str.split(',')]

    # ── Real inference: filter ALL candidate positions ──
    y_pred_new_local = np.zeros(seq_len)

    for i in range(seq_len):
        if user_preds[i] == 0:
            continue  # not a candidate → skip

        # This is a candidate (your model predicted positive)
        avg_prob = user_probs[i]  # default: at least your prob
        if lms_probs is not None and i < len(lms_probs):
            avg_prob = USER_WEIGHT * user_probs[i] + LMS_WEIGHT * lms_probs[i]

        if avg_prob >= A:
            y_pred_new_local[i] = 1

    global_y_pred_new.extend(y_pred_new_local)

# ────────────────────────────────────────────────────────────────
# Evaluation (only if labels were present)
# ────────────────────────────────────────────────────────────────
global_y_true = np.array(global_y_true) if global_y_true else None
global_y_pred_new = np.array(global_y_pred_new)

if global_y_true is not None and len(global_y_true) > 0:
    TN, FP, FN, TP = confusion_matrix(global_y_true, global_y_pred_new).ravel()

    Sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    Spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    Acc  = (TP + TN) / len(global_y_true)
    Prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    F1   = 2 * Prec * Sens / (Prec + Sens) if (Prec + Sens) > 0 else 0.0
    MCC  = matthews_corrcoef(global_y_true, global_y_pred_new)

    print("\nEvaluation on labeled test set (simulating real inference):")
    print(f"  TP: {TP:6d}")
    print(f"  FP: {FP:6d}")
    print(f"  TN: {TN:6d}")
    print(f"  FN: {FN:6d}")
    print(f"Sensitivity: {Sens:.4f}")
    print(f"Specificity: {Spec:.4f}")
    print(f"Accuracy:    {Acc:.4f}")
    print(f"Precision:   {Prec:.4f}")
    print(f"F1-score:    {F1:.4f}")
    print(f"MCC:         {MCC:.4f}")
else:
    print("\nReal prediction mode: no labels available → only final predictions generated.")

Real-world inference mode: filter ALL candidate positives
Weights → User: 0.10 | LMetalSite: 0.90
Threshold A = 0.026064

Evaluation on labeled test set (simulating real inference):
  TP:    185
  FP:    584
  TN:  19610
  FN:     40
Sensitivity: 0.8222
Specificity: 0.9711
Accuracy:    0.9694
Precision:   0.2406
F1-score:    0.3722
MCC:         0.4350


# ZN Binding

In [23]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import confusion_matrix, matthews_corrcoef

# ────────────────────────────────────────────────────────────────
# Settings — same as before
# ────────────────────────────────────────────────────────────────
USER_WEIGHT = 0.1
LMS_WEIGHT  = 0.9
# A = 0.24803199999999997
A = 0.006226
# A = 0.42 # for ZN ion
df_user = pd.read_csv('ZN_metal_per_protein_predictions.csv')
df_lms  = pd.read_csv('ZN_Test.bind.csv', skiprows=1)

print(f"Real-world inference mode: filter ALL candidate positives")
print(f"Weights → User: {USER_WEIGHT:.2f} | LMetalSite: {LMS_WEIGHT:.2f}")
print(f"Threshold A = {A}")

global_y_true = []          # only used if labels exist (for evaluation)
global_y_pred_new = []

for _, row in df_user.iterrows():
    pid = row['ID']
    sequence = row['sequence']
    seq_len = len(sequence)

    # Parse your model's predictions
    user_probs = [float(p) for p in row['prob'].split(',')]
    user_preds = [int(p) for p in row['pred'].split(',')]

    # If labels exist (for evaluation only)
    has_labels = 'TP' in row and 'FP' in row
    if has_labels:
        tp_list = ast.literal_eval(row['TP'])
        fn_list = ast.literal_eval(row['FN'])
        y_true_local = np.zeros(seq_len)
        y_true_local[tp_list + fn_list] = 1
        global_y_true.extend(y_true_local)
    else:
        y_true_local = None  # no labels → real prediction mode

    # Get LMetalSite probs
    lms_probs = None
    lms_row = df_lms[df_lms['ID'] == pid]
    if not lms_row.empty:
        lms_probs_str = lms_row['Mn2+ binding prob'].values[0]
        lms_probs = [float(p.strip()) for p in lms_probs_str.split(',')]

    # ── Real inference: filter ALL candidate positions ──
    y_pred_new_local = np.zeros(seq_len)

    for i in range(seq_len):
        if user_preds[i] == 0:
            continue  # not a candidate → skip

        # This is a candidate (your model predicted positive)
        avg_prob = user_probs[i]  # default: at least your prob
        if lms_probs is not None and i < len(lms_probs):
            avg_prob = USER_WEIGHT * user_probs[i] + LMS_WEIGHT * lms_probs[i]

        if avg_prob >= A:
            y_pred_new_local[i] = 1

    global_y_pred_new.extend(y_pred_new_local)

# ────────────────────────────────────────────────────────────────
# Evaluation (only if labels were present)
# ────────────────────────────────────────────────────────────────
global_y_true = np.array(global_y_true) if global_y_true else None
global_y_pred_new = np.array(global_y_pred_new)

if global_y_true is not None and len(global_y_true) > 0:
    TN, FP, FN, TP = confusion_matrix(global_y_true, global_y_pred_new).ravel()

    Sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    Spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    Acc  = (TP + TN) / len(global_y_true)
    Prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    F1   = 2 * Prec * Sens / (Prec + Sens) if (Prec + Sens) > 0 else 0.0
    MCC  = matthews_corrcoef(global_y_true, global_y_pred_new)

    print("\nEvaluation on labeled test set (simulating real inference):")
    print(f"  TP: {TP:6d}")
    print(f"  FP: {FP:6d}")
    print(f"  TN: {TN:6d}")
    print(f"  FN: {FN:6d}")
    print(f"Sensitivity: {Sens:.4f}")
    print(f"Specificity: {Spec:.4f}")
    print(f"Accuracy:    {Acc:.4f}")
    print(f"Precision:   {Prec:.4f}")
    print(f"F1-score:    {F1:.4f}")
    print(f"MCC:         {MCC:.4f}")
else:
    print("\nReal prediction mode: no labels available → only final predictions generated.")

Real-world inference mode: filter ALL candidate positives
Weights → User: 0.10 | LMetalSite: 0.90
Threshold A = 0.006226

Evaluation on labeled test set (simulating real inference):
  TP:    940
  FP:   2968
  TN:  52013
  FN:     99
Sensitivity: 0.9047
Specificity: 0.9460
Accuracy:    0.9453
Precision:   0.2405
F1-score:    0.3800
MCC:         0.4506
